# Research Framework and Book Structure

## Overview of Contributions and Methodological Progression

## 1. Research Objective

This book investigates machine learning methods for accelerating electromagnetic device design. The work progresses through two complementary paradigms: supervised deep learning for rapid field and perfromance maps prediction (Chapters 2-3), and reinforcement learning for autonomous design optimization (Chapters 4-5). These approaches address different stages of the design cycle while sharing a common objective: reducing computational barriers that constrain design space exploration. In the experiments of this book the trained surrogates evaluate a design two orders of magnitude faster than the finite element solve on the same laptop (16 ms against 2.1 s in Chapter 2), at 0.17% normalised mean squared error for field maps (Chapter 2) and a cross-validated R² of 0.91–0.97 for efficiency maps (Chapter 3).

## 2. The Narrative of Chapters

### Chapter 2: Deep Learning for Magnetic Field Distribution Prediction

Chapter 2 establishes the foundation for machine learning-based electromagnetic modeling through two complementary approaches: data-driven convolutional neural networks and physics-informed neural networks. Its data pipeline simulates three problems of increasing complexity — a coil in air, a transformer with a nonlinear magnetic core, and an interior permanent magnet motor — and the learning experiments use the coil: 1,000 FEMM solutions, a U-Net whose dilated encoder convolutions capture the long-range spatial dependencies dictated by electromagnetic field theory, 0.17% normalized mean squared error on held-out field maps, and a prediction cost of 16 ms against 2.1 s per finite element solve. Monte Carlo dropout is implemented for uncertainty estimates and its calibration is measured rather than assumed (the nominal 90% interval covered 52% of the test points on a synthetic task). Physics-informed neural networks (PINNs) add the magnetostatic Poisson equation to the loss and reach 0.14% error from 100 field observations of one coil, against 15.8% for the same network trained on the data alone, with an ablation of each ingredient. This dual methodology shows that both data-driven learning (high accuracy with sufficient data) and physics-constrained learning (data efficiency) accelerate electromagnetic field prediction when designed with domain-aware choices — in the PINN's case, giving the network the radial coordinates the solution is built from.

### Chapter 3: Recurrent Neural Networks for Multi-Objective Performance Map Prediction

Building on the spatial field prediction of Chapter 2, Chapter 3 moves from instantaneous field distributions to complete performance maps spanning the torque–speed operating envelope — the quantity needed to evaluate a motor over drive cycles rather than at isolated operating points. A FEMM pipeline characterises each interior permanent magnet design over tens of operating points (19–22 s per design) and produced 2,234 valid designs of 2,800 sampled, plus 60 designs of a fractional-slot concentrated-winding variant. The chapter reviews recurrent and attention-based sequence models and trains a map decoder — a small multilayer perceptron on the design parameters feeding a transposed-convolution decoder of the 60×80 efficiency map — reaching a cross-validated R² of 0.912 for the baseline and 0.971 for the widest variant. Transfer-learning experiments measure when a model trained on one task accelerates learning on another: fine-tuning a pretrained model with its output layer reset beat training from scratch at the smallest data sizes (+0.05 R² at 10 designs for the power-factor maps) and, for the widest model, at every size tested. The chapter thus answers "how well does the machine perform" across its operating space, the evaluation that design optimization algorithms call thousands of times.

### Chapter 4: Sequential Topology Optimization through Markov Decision Process Formulation

Chapter 4 marks a methodological transition from supervised prediction tasks to optimization-driven design generation. Rather than applying machine learning methods directly to the topology optimization problem, this chapter reformulates the problem itself — a C-core actuator and synchronous reluctance motor flux barrier design — as a sequential decision-making process. The contribution lies primarily in problem formulation: by representing material deposition as a Markov Decision Process where design proceeds through sequential placement decisions rather than simultaneous density assignment across all elements, the work eliminates persistent artifacts (checkerboard patterns, isolated material islands, thin connections) that plague conventional density-based topology optimization and require extensive post-processing filtering. This MDP formulation employs minimal machine learning during its development phase; instead, it establishes the mathematical infrastructure and physics-based simulation environment that enables the reinforcement learning investigations in Chapter 5. Tabular Q-learning and a genetic algorithm run on this environment with a FEMM solve at every step, and the chapter examines whether the sequential formulation produces connected, manufacturable topologies without penalty terms or filtering operations. The chapter provides the essential groundwork—the "stage" on which reinforcement learning agents will perform.

### Chapter 5: Deep Reinforcement Learning Agents for Autonomous Design

Chapter 5 completes the research arc by introducing deep reinforcement learning agents that operate within the MDP framework established in Chapter 4. Where Chapters 2 and 3 required extensive pre-computed datasets from finite element simulations (supervised learning's data dependency), reinforcement learning agents learn directly through interaction with the physics-based simulation environment, treating electromagnetic analysis as an oracle that provides reward signals rather than training labels. Deep Q-networks, advantage actor-critic and proximal policy optimisation agents, implemented in PyTorch, discover design strategies through trial-and-error experimentation, guided by reward functions that encode engineering objectives such as force or torque production. The chapter trains these agents for a fixed budget of episodes with a FEMM solve at every step (the deep Q-network and actor-critic runs took 10.8 h and 6.2 h on a laptop) and compares what each reached against the Q-learning and genetic-algorithm baselines of Chapter 4, without any human-curated training dataset. This represents a fundamentally different paradigm from supervised learning: rather than learning to predict outputs from example inputs (Chapters 2-3), the agent learns to generate designs through experimentation, developing strategies that improve with experience. The chapter shows that the MDP formulation from Chapter 4—with its sequential structure and well-defined state-action-reward mechanics—provides the necessary framework for reinforcement learning to operate effectively in electromagnetic design.

### The Methodological Arc: From Prediction to Autonomous Design

Across these investigations, a clear methodological progression emerges that divides the work into two complementary phases. Chapters 2 and 3 focus on deep learning for performance prediction, treating electromagnetic simulation as a supervised learning problem: given geometry and operating conditions as inputs, predict field distributions (Chapter 2) or performance metrics (Chapter 3) as outputs. These data-driven surrogates accelerate existing design workflows by replacing expensive simulations with rapid neural network inference, enabling optimization algorithms that require massive numbers of performance evaluations. The supervised learning chapters show that convolutional and decoder architectures can approximate the field and performance maps of the problems studied with accuracies of 0.17% NMSE and R² up to 0.97. Chapters 4 and 5 pivot toward design generation, where the objective shifts from prediction to discovery of optimal geometries. Chapter 4 establishes the sequential decision formulation that makes autonomous design tractable—transforming a high-dimensional simultaneous optimization problem into a series of lower-dimensional sequential decisions. Chapter 5 then tests whether reinforcement learning agents can learn this sequential design process through interaction with the environment, which actions lead to high-performing designs, and at what cost in solver calls, without labeled examples. Together, these complementary approaches—predictive surrogates that accelerate evaluation (Chapters 2-3) and autonomous agents that learn to design (Chapters 4-5)—provide a comprehensive framework for machine learning in electromagnetic design, addressing both rapid analysis and intelligent synthesis.

## 3. Mathematical and Computational Prerequisites

The mathematical foundations required include multivariate calculus (partial derivatives, gradients, chain rule), linear algebra (vector spaces, matrix operations, eigenvalues), probability theory (expectation, variance, conditional distributions), and introductory electromagnetic theory (Maxwell's equations, magnetic circuits). Familiarity with numerical methods, particularly finite element analysis concepts, proves beneficial though not strictly necessary as relevant physics is introduced where needed.

Computational implementations throughout employ Python 3.12, with PyTorch 2.14 as the deep learning framework. Electromagnetic simulations use FEMM 5 (the open-source femm-qt build of Finite Element Method Magnetics), driven from Python on Linux. Network architectures and training loops are written in PyTorch from basic modules to ensure transparency and educational value. The computational requirements are modest: every result in this book was produced on one laptop — an Intel Core i7-11370H with 14 GB of memory and an NVIDIA GeForce MX450 with 2 GB — except one Chapter 2 training run on a free cloud T4 GPU. Training durations range from seconds (the force network of Chapter 2; 110 s for the PINN) through tens of minutes (36 min per U-Net) and hours (Chapter 3's twelve training runs, about 5 h in total) to the reinforcement-learning agents of Chapter 5, which spend most of their 6–11 h per run inside the finite element solver.

Standard mathematical notation applies throughout: scalars in italics (*x*, *y*, *θ*), vectors in bold lowercase (**x**, **H**, **B**), matrices in bold uppercase (**W**, **X**), sets in calligraphic font (𝒟, ℒ), and operators following conventional definitions (∇ for gradient, ∇· for divergence, ∇× for curl). Electromagnetic quantities follow SI units with standard symbols: **H** for magnetic field intensity (A/m), **B** for magnetic flux density (T), **J** for current density (A/m²), μ for permeability (H/m).

## 4. Structure and Dependencies

The chapters exhibit both sequential dependencies and parallel development paths. Chapter 2 must precede Chapter 3, as the latter builds directly on convolutional architectures and training methodologies established for field prediction, extending them to performance map sequences. Chapters 4 and 5 form a tightly coupled pair, with Chapter 5's reinforcement learning agents operating within Chapter 4's MDP environment—the state space, action space, and transition dynamics defined in Chapter 4 become the world in which Chapter 5's agents learn. However, the two main research threads—supervised learning (Chapters 2-3) and sequential optimization (Chapters 4-5)—develop largely independently and can be studied in parallel. Transfer learning concepts introduced in Chapter 3 inform the generalization discussions in Chapter 5, providing a connecting thread between the otherwise distinct methodological paths. For readers primarily interested in supervised learning applications, Chapters 2 and 3 form a coherent unit. Those focused on optimization and reinforcement learning may concentrate on Chapters 4 and 5. Reading sequentially from Chapter 2 through Chapter 5 provides the complete narrative of progressing from prediction to autonomous design, revealing how machine learning transforms both analysis and synthesis in electromagnetic engineering.

## References

```{bibliography}
:filter: docname in docnames
```